In [14]:
import torch
from transformers import RobertaTokenizer, RobertaConfig, RobertaModel, AutoTokenizer, AutoModelForSequenceClassification, AutoModelForCausalLM, Trainer, TrainingArguments
from tqdm import tqdm
import os
from datasets import Dataset
import random
from sklearn.metrics import precision_recall_fscore_support, accuracy_score, confusion_matrix, classification_report
import numpy as np

os.environ["CUDA_LAUNCH_BLOCKING"] = "1"
os.environ["CUDA_VISIBLE_DEVICES"] = ""
DATASET_ROOT = "../../CrossVul"
ALLOWED_CWE_IDS = {"CWE-79"} # "CWE-79", "CWE-89", "CWE-787"
LANGUAGES = ['c', 'cpp', 'cs', 'html', 'java', 'py', 'php']
SEED = 42

In [15]:
vulBERTa = "claudios/VulBERTa-MLP-ReVeal"
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
tokenizer = AutoTokenizer.from_pretrained(vulBERTa, trust_remote_code=True)
print(device)

cuda


# Model Finetuning:

In [23]:
def chunk_code(code, chunk_size=512, stride=256):
    tokens = tokenizer(code, return_attention_mask=False, return_token_type_ids=False, truncation=False)["input_ids"]
    chunks = []
    for i in range(0, len(tokens), stride):
        chunk = tokens[i:i+chunk_size]
        if len(chunk) < chunk_size:
            chunk += [tokenizer.pad_token_id] * (chunk_size - len(chunk))
        chunks.append(chunk)
        if i + chunk_size >= len(tokens):
            break
    return chunks

def predict_with_chunk_voting(trainer, dataset, chunk_size=512, stride=256):
    true_labels = []
    pred_labels = []

    for example in dataset:
        code = example["code"]
        label = example["label"]
        true_labels.append(label)

        chunks = chunk_code(code, chunk_size=chunk_size, stride=stride)

        input_ids = torch.tensor(chunks).to(trainer.model.device)
        attention_mask = (input_ids != tokenizer.pad_token_id).long()

        with torch.no_grad():
            outputs = trainer.model(input_ids=input_ids, attention_mask=attention_mask)
            logits = outputs.logits
            preds = torch.argmax(logits, dim=1).cpu().numpy()

        file_pred = 1 if (preds.mean() > 0.2) else 0
        pred_labels.append(file_pred)

    return true_labels, pred_labels

def collect_files_for_cwe(cwe_id):
    samples = []
    for lang in LANGUAGES:
        lang_dir = os.path.join(DATASET_ROOT, cwe_id, lang)
        if not os.path.isdir(lang_dir):
            continue
        for filename in os.listdir(lang_dir):
            filepath = os.path.join(lang_dir, filename)
            if filename.endswith('.DS_Store'):
                continue
            label = 1 if "bad" in filename.lower() else 0
            with open(filepath, 'r', encoding='utf-8', errors='ignore') as f:
                code = f.read()
            samples.append({
                "filename": filename,
                "code": code,
                "label": label
            })
    return samples

def tokenize_example(example):
    prompt = "Does this code have a vulnerability relating to CWE-22 Path Traversal?"
    encoded = tokenizer(
        example["code"],
        padding="max_length",
        max_length=512,
        truncation=True
    )
    return {
        "input_ids": encoded["input_ids"],
        "attention_mask": encoded["attention_mask"],
        "label": example["label"]
    }

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    precision, recall, f1, _ = precision_recall_fscore_support(labels, preds, average='binary', zero_division=0)
    acc = accuracy_score(labels, preds)
    return {
        'accuracy': acc,
        'precision': precision,
        'recall': recall,
        'f1': f1,
    }

In [ ]:
for cwe_id in ALLOWED_CWE_IDS:
    print(f"\n--- Training for {cwe_id} ---")
    print(cwe_id)
    samples = collect_files_for_cwe(cwe_id)
    
    random.seed(SEED)
    random.shuffle(samples)

    raw_dataset = Dataset.from_list(samples)
    tokenized_dataset = raw_dataset.map(tokenize_example)

    train_test = tokenized_dataset.train_test_split(test_size=0.2)
    train_dataset = train_test["train"]
    eval_dataset = train_test["test"]

    model_path = f"./models/vulberta_{cwe_id}/final"

    # if os.path.exists(model_path):
    #     print(f"Loading existing model for {cwe_id}...")
    #     model = AutoModelForSequenceClassification.from_pretrained(model_path).to(device)
    # else:
    print(f"Training new model for {cwe_id}...")
    model = AutoModelForSequenceClassification.from_pretrained(vulBERTa, num_labels=2).to(device)

    training_args = TrainingArguments(
        output_dir=f"./models/vulberta_{cwe_id}",
        evaluation_strategy="epoch",
        learning_rate=2e-5,
        per_device_train_batch_size=8,
        per_device_eval_batch_size=8,
        num_train_epochs=4,
        weight_decay=0.01,
        save_strategy="epoch",
        load_best_model_at_end=True,
        metric_for_best_model="eval_loss",
        remove_unused_columns = False
    )

    trainer = Trainer(
        model=model,
        args=training_args,
        train_dataset=train_dataset,
        eval_dataset=eval_dataset,
        compute_metrics=compute_metrics,
    )

    trainer.train()
    trainer.save_model(model_path)

    trainer = Trainer(
        model=model,
        args=TrainingArguments(output_dir="./tmp"),
        eval_dataset=   eval_dataset,
        compute_metrics=compute_metrics
    )

    true_labels, pred_labels = predict_with_chunk_voting(trainer, eval_dataset)
    precision, recall, f1, _ = precision_recall_fscore_support(true_labels, pred_labels, average='binary', zero_division=0)
    acc = accuracy_score(true_labels, pred_labels)

    print(f"Metrics for {cwe_id}:")
    print({
        'accuracy': acc,
        'precision': precision,
        'recall': recall,
        'f1': f1,
    })

    print(f"\nConfusion Matrix for {cwe_id}:")
    print(confusion_matrix(true_labels, pred_labels))



--- Training for CWE-79 ---
CWE-79


Map:   0%|          | 0/2142 [00:00<?, ? examples/s]

Training new model for CWE-79...


C:\Users\robpi\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.12_qbz5n2kfra8p0\LocalCache\local-packages\Python312\site-packages\transformers\training_args.py:1575: FutureWarning: `evaluation_strategy` is deprecated and will be removed in version 4.46 of 🤗 Transformers. Use `eval_strategy` instead
  warnings.warn(
